In [0]:
import pyspark.sql.functions as F

catalog_name = "f1_dev"
target_table = f"{catalog_name}.gold.ferrari_stock_race_impact"


In [0]:
# 1. Load Silver Tables
df_results = spark.table(f"{catalog_name}.silver.fact_results")
df_constructors = spark.table(f"{catalog_name}.silver.dim_constructors")
df_races = spark.table(f"{catalog_name}.silver.dim_races")
df_stock = spark.table(f"{catalog_name}.silver.dim_ferrari_stock")



In [0]:
# 2. Filter & Aggregate Ferrari Race Performance (Restricted to >= 2016-10-05)

# Define the stock dataset start threshold
STOCK_START_DATE = "2016-10-05"

df_ferrari_races = (
    df_results
    .join(df_constructors, "constructor_id")
    .join(df_races, "race_id")
    .filter(F.lower(F.col("constructor_name")).contains("ferrari"))
    .filter(F.col("race_date") >= F.to_date(F.lit(STOCK_START_DATE)))
    .groupBy("race_id", "race_year", "race_round", "race_name", "race_date")
    .agg(
        F.min("position").alias("best_finish"),
        F.sum("points").alias("total_points"),
        F.max(F.when(F.col("position") == 1, 1).otherwise(0)).alias("is_win"),
        F.max(F.when(F.col("position") <= 3, 1).otherwise(0)).alias("is_podium"),
        F.max(F.when(F.col("points") == 0, 1).otherwise(0)).alias("is_zero_points")
    )
)



In [0]:
# 3. Derive Alignment Dates for Pre-Race Friday and Post-Race Monday
df_race_windows = (
    df_ferrari_races
    .withColumn("friday_pre_race", F.date_sub(F.col("race_date"), 2))
    .withColumn("monday_post_race", F.date_add(F.col("race_date"), 1))
)



In [0]:
# 4. Join Stock Data for Friday Pre-Race and Monday Post-Race Trading Days
df_stock_fri = df_stock.select(
    F.col("trade_date").alias("friday_pre_race"),
    F.col("close_price").alias("fri_close_price")
)

df_stock_mon = df_stock.select(
    F.col("trade_date").alias("monday_post_race"),
    F.col("open_price").alias("mon_open_price"),
    F.col("close_price").alias("mon_close_price"),
    F.col("volume").alias("mon_volume")
)

df_impact_raw = (
    df_race_windows
    .join(df_stock_fri, "friday_pre_race", "inner")
    .join(df_stock_mon, "monday_post_race", "inner")
)



In [0]:
# 5. Calculate Percentage Returns & Categorize Race Outcome
df_gold_impact = (
    df_impact_raw
    .withColumn(
        "weekend_gap_pct", 
        F.round(((F.col("mon_open_price") - F.col("fri_close_price")) / F.col("fri_close_price")) * 100, 2)
    )
    .withColumn(
        "total_post_race_return_pct", 
        F.round(((F.col("mon_close_price") - F.col("fri_close_price")) / F.col("fri_close_price")) * 100, 2)
    )
    .withColumn(
        "result_category", 
        F.when(F.col("is_win") == 1, "1_Win")
         .when(F.col("is_podium") == 1, "2_Podium")
         .when(F.col("is_zero_points") == 1, "4_Zero_Points_DNF")
         .otherwise("3_Points_Finish")
    )
    .select(
        "race_year",
        "race_round",
        "race_name",
        "race_date",
        "result_category",
        "best_finish",
        "total_points",
        "friday_pre_race",
        "fri_close_price",
        "monday_post_race",
        "mon_open_price",
        "mon_close_price",
        "weekend_gap_pct",
        "total_post_race_return_pct",
        "mon_volume"
    )
)



In [0]:
# 6. Save Data Product to Gold Schema
(
    df_gold_impact.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)



In [0]:
# 7. Apply Unity Catalog Governance Tags
spark.sql(f"""
    ALTER TABLE {target_table} SET TAGS (
        'layer' = 'gold',
        'data_product' = 'ferrari_stock_race_impact',
        'domain' = 'financial_analytics',
        'start_date_bound' = '{STOCK_START_DATE}'
    )
""")

print(f"✓ Data product built successfully with date filter >= {STOCK_START_DATE}: {target_table}")